In [2]:
import time
import os, sys, signal
import numpy as np
from pathlib import Path
import random
import signal
from signal import Signals
from threading import Thread, Condition
import uuid

from concurrent.futures import ProcessPoolExecutor as Exe
from metasmith.coms.via_file_watcher import RemoteShell
from metasmith.coms.terminals import CurrentTimeMillis, TerminalProcess, RemoveTrailingNewline
from metasmith.coms.ipc import GenerateId, ResetGenerator
from local.constants import WORKSPACE_ROOT

In [3]:
''.join(("%012X" % uuid.getnode())[i:i+2] for i in range(0, 12, 2))

'00155D2A3512'

In [4]:
hex(uuid.getnode())

'0x155d2a3512'

In [5]:
relay_path = WORKSPACE_ROOT/"main/relay_agent/XPS-laptop"

In [6]:
# with RemoteShell(relay_path) as shell:
#     res = shell.Exec(f"echo 1", history=True, timeout=3)
# print(res)

In [7]:
with RemoteShell(relay_path) as shell:
    shell.RegisterOnOut(print)
    res = shell.Exec(
        f"""\
            pwd -P
        """,
        history=True,
        timeout=5,
    )
# print(res)

/home/tony/workspace/tools/Metasmith/main/relay_agent


In [7]:
res.out

['Current counter value: 0\n',
 '',
 '',
 '',
 'Current counter value: 1\n',
 'Current counter value: 2\n',
 '',
 'Current counter value: 3\n']

In [10]:
from threading import Condition
lock = Condition()

def r(args):
    ResetGenerator()
    i = args
    salt =  GenerateId(32)
    for retry in range(100):
        try:
            with RemoteShell(relay_path) as shell:
                res = shell.Exec(f"echo {salt}", history=True, timeout=3)
                if len(res.out)!=1 or salt not in res.out:
                    return i, res.out
                else:
                    return i, None
        except (TimeoutError, ConnectionError, FileNotFoundError):
            dt = random.random()*(2**(min(retry, 5)-5))
            # pri
            # nt(f"[{i}] timeout, retry in [{dt}]s")
            time.sleep(dt)
        except KeyboardInterrupt:
            return False, None
    return False, None
mypid = os.getpid()
fd_path = Path(f"/proc/{mypid}/fd")
n_fd = len(list(fd_path.iterdir()))
# print(f"start: {n_fd}")
# k, c = 1000, 14
k, c = 100, 4
# k, c = 14, 14
failed = 0
results = []
with Exe(max_workers=c) as exe:
    for res in exe.map(r, list(range(k))):
        n_fd = len(list(fd_path.iterdir()))
        # print(f"fd: {n_fd}", end="\r")
        results.append(res)
        i, o = res
        # if o is not None:
        #     print(o)
print()
print(len(results))
# for out in results:
#     print(out)

# for i in range(k):
#     r(i)
#     print(i, end="\r")


100
